In [97]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

In [98]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv("data/test.csv")

df_test["Transported"] = False
df = pd.concat([df_train, df_test])
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
1097,2330_01,Earth,False,F/473/P,PSO J318.5-22,21.0,False,17.0,0.0,0.0,595.0,0.0,Jarena Weaverays,False
398,0838_02,Mars,False,D/35/P,TRAPPIST-1e,23.0,False,2577.0,0.0,0.0,0.0,2.0,Chal Concy,False
6687,7052_01,Europa,NaN,E/469/S,55 Cancri e,27.0,False,0.0,0.0,0.0,0.0,0.0,Terons Stranbeate,True
6102,6444_02,Earth,True,G/1041/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Elicey Domington,True
4071,4348_01,Europa,True,B/142/P,TRAPPIST-1e,47.0,True,0.0,0.0,0.0,0.0,0.0,Krazet Conate,True


In [99]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='str')

In [100]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  str    
 4   Destination   12696 non-null  str    
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  str    
 13  Transported   12970 non-null  bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.4+ MB


In [101]:
df.isnull().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
Transported       0
dtype: int64

In [102]:
df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,12700.000000,12707.000000,12681.000000,12664.000000,12686.000000,12702.000000
mean,28.771969,222.897852,451.961675,174.906033,308.476904,306.789482
std,14.387261,647.596664,1584.370747,590.558690,1130.279641,1180.097223
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,77.000000,29.000000,57.000000,42.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [103]:
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
2819,3039_01,Europa,False,A/37/S,TRAPPIST-1e,32.0,False,0.0,0.0,0.0,0.0,0.0,Acraban Conamminal,False
1081,1149_03,Earth,True,G/178/S,TRAPPIST-1e,3.0,False,0.0,0.0,0.0,0.0,0.0,Sonnie Quinnerettt,False
6698,7069_02,Earth,True,G/1159/S,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Essara Kellynn,False
3460,3724_01,Earth,True,G/609/S,TRAPPIST-1e,24.0,False,0.0,0.0,0.0,0.0,0.0,Yvonna Mcdowns,True
3112,3353_06,Europa,False,C/125/S,TRAPPIST-1e,45.0,False,0.0,1897.0,0.0,880.0,605.0,Hamelik Reming,True


In [104]:
df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split("/", expand=True)

In [105]:
df.drop(columns="Cabin", inplace=True)
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
2264,2429_01,Earth,False,TRAPPIST-1e,17.0,False,112.0,56.0,332.0,0.0,606.0,Idace Vinozarks,False,F,497,P
934,0995_01,Europa,True,55 Cancri e,46.0,False,0.0,0.0,0.0,0.0,0.0,Chabik Barmant,True,C,36,P
461,0959_01,Earth,False,PSO J318.5-22,24.0,False,640.0,155.0,0.0,0.0,0.0,Dixiel Franankson,False,E,71,S
5096,5445_01,NaN,False,TRAPPIST-1e,61.0,False,0.0,53.0,813.0,3.0,0.0,Min Alfordonard,False,F,1125,P
6501,6862_01,Earth,True,TRAPPIST-1e,48.0,False,0.0,0.0,0.0,0.0,0.0,Diandi Mcfaddennon,True,G,1115,S


In [106]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12700 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12707 non-null  float64
 7   FoodCourt     12681 non-null  float64
 8   ShoppingMall  12664 non-null  float64
 9   Spa           12686 non-null  float64
 10  VRDeck        12702 non-null  float64
 11  Name          12676 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [107]:
imputer = KNNImputer(n_neighbors=5, weights="distance")
num_cols = df[["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa","VRDeck"]]
filled_data = imputer.fit_transform(num_cols)

In [108]:
filled_data = pd.DataFrame(filled_data, columns=num_cols.columns, index=num_cols.index)
filled_data

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,39.000000,0.0,0.0,0.0,0.0,0.0
1,24.000000,109.0,9.0,25.0,549.0,44.0
2,58.000000,43.0,3576.0,0.0,6715.0,49.0
3,33.000000,0.0,1283.0,371.0,3329.0,193.0
4,16.000000,303.0,70.0,151.0,565.0,2.0
...,...,...,...,...,...,...
4272,34.000000,0.0,0.0,0.0,0.0,0.0
4273,42.000000,0.0,847.0,17.0,10.0,144.0
4274,35.200000,0.0,0.0,0.0,0.0,0.0
4275,23.666667,0.0,2680.0,0.0,0.0,523.0


In [109]:
df[num_cols.columns] = filled_data

In [110]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP             296
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            294
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [111]:
df.sample(10)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
292,0322_01,Earth,True,PSO J318.5-22,13.0,False,0.0,0.0,0.0,0.0,0.0,Annya Blange,False,G,45,S
856,0922_02,Europa,False,55 Cancri e,31.0,False,0.0,1095.0,0.0,169.0,1226.0,Achyon Obedaming,False,B,35,P
2717,2910_01,Earth,False,PSO J318.5-22,32.0,False,0.0,0.0,736.0,0.0,0.0,Celis Bucknersony,False,F,602,P
2920,6423_01,Europa,False,55 Cancri e,55.0,False,1.0,9322.0,26.0,12690.0,137.0,Edar Aginoid,False,C,239,S
649,0680_03,Earth,True,TRAPPIST-1e,5.0,NaN,0.0,0.0,0.0,0.0,0.0,Joandy Camerrison,False,G,105,P
2155,2306_06,Europa,False,55 Cancri e,47.0,False,1320.0,428.0,4.0,24.0,0.0,Luxons Colensid,False,C,82,P
6365,6731_01,Earth,False,TRAPPIST-1e,32.0,False,3135.0,2.0,0.0,0.0,0.0,Tin Rochan,False,F,1397,P
2136,2291_03,Earth,False,55 Cancri e,46.0,False,426.0,37.0,0.0,0.0,568.0,NaN,False,F,445,S
1179,2482_02,Europa,False,55 Cancri e,40.0,False,33.0,3030.0,0.0,3588.0,377.0,Lesat Nairconed,False,D,84,S
68,0072_01,Earth,False,TRAPPIST-1e,14.0,False,0.0,1.0,0.0,0.0,1063.0,Thell Brantuarez,False,F,17,S


In [112]:
df["Name"] = df["Name"].fillna('U')

In [113]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12970 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12970 non-null  float64
 7   FoodCourt     12970 non-null  float64
 8   ShoppingMall  12970 non-null  float64
 9   Spa           12970 non-null  float64
 10  VRDeck        12970 non-null  float64
 11  Name          12970 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [114]:
df['CryoSleep'].value_counts()

CryoSleep
False    8079
True     4581
Name: count, dtype: int64

In [115]:
df['Destination'].value_counts()

Destination
TRAPPIST-1e      8871
55 Cancri e      2641
PSO J318.5-22    1184
Name: count, dtype: int64

In [116]:
df['HomePlanet'].value_counts()

HomePlanet
Earth     6865
Europa    3133
Mars      2684
Name: count, dtype: int64

In [117]:
df["VIP"].value_counts()

VIP
False    12401
True       273
Name: count, dtype: int64

In [118]:
df["VIP"] = df["VIP"].fillna(False)

In [119]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name              0
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [120]:
df['CryoSleep'].value_counts()

CryoSleep
False    8079
True     4581
Name: count, dtype: int64

In [121]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name',
       'Transported', 'Deck', 'Num', 'Side'],
      dtype='str')

In [122]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df["CryoSleep"] = df["CryoSleep"].fillna(
    (df[spend_cols].sum(axis=1) == 0)
)

In [123]:
df['Deck'].value_counts(), df['Num'].value_counts(), df['Side'].value_counts()

(Deck
 F    4239
 G    3781
 E    1323
 B    1141
 C    1102
 D     720
 A     354
 T      11
 Name: count, dtype: int64,
 Num
 82      34
 4       28
 56      28
 31      27
 95      27
         ..
 1882     1
 1883     1
 1885     1
 1887     1
 1890     1
 Name: count, Length: 1894, dtype: int64,
 Side
 S    6381
 P    6290
 Name: count, dtype: int64)

In [124]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12970 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12970 non-null  float64
 5   VIP           12970 non-null  object 
 6   RoomService   12970 non-null  float64
 7   FoodCourt     12970 non-null  float64
 8   ShoppingMall  12970 non-null  float64
 9   Spa           12970 non-null  float64
 10  VRDeck        12970 non-null  float64
 11  Name          12970 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [125]:
df['Group'] = df["PassengerId"].str.split("_").str[0]

In [126]:
df["Group"].value_counts()

Group
0984    8
4005    8
4256    8
4498    8
5133    8
       ..
9265    1
9269    1
9271    1
9273    1
9277    1
Name: count, Length: 9280, dtype: int64

In [127]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep         0
Destination     274
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name              0
Transported       0
Deck            299
Num             299
Side            299
Group             0
dtype: int64

In [128]:
df['Deck'] = df.groupby("Group")['Deck'].transform(
    lambda x : x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
)

In [129]:
df['Deck'] = df["Deck"].fillna(df["Deck"].mode()[0])
df["Side"] = df["Side"].fillna(df["Side"].mode()[0])

In [130]:
df.drop(columns='Num', inplace=True)

In [131]:
df['HomePlanet'] = df.groupby(["Group", "Deck", "Side"])["HomePlanet"].transform(
    lambda x:x.fillna(
        x.mode().iloc[0] if not x.mode().empty else np.nan
        ))

In [135]:
df["HomePlanet"] = df["HomePlanet"].fillna(df["HomePlanet"].mode()[0])

In [137]:
df["Destination"] = df.groupby(["Group", "HomePlanet"])["Destination"].transform(
    lambda x : x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
)

In [139]:
df["Destination"] = df["Destination"].fillna(df["Destination"].mode()[0])

In [140]:
df.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
Deck            0
Side            0
Group           0
dtype: int64